# Contract-driven charging protocol

An executable prototype of [PROTOCOL.md](../docs/PROTOCOL.md). One charging
point carries $job:X\to\mathbb N_0$, where $0$ means unallocated. A vehicle asks
for a new job $k>0$ over a duration $d>0$, starting anywhere in a bounded start
set $I$:

$$
C^{\mathrm{req}}=\left\langle\top\;\middle|\;\Diamond_I\Box_d(job=k)\right\rangle.
$$

The protocol turns that into concrete offer contracts
$\left\langle\top\mid\Box_{[s,s+d)}(job=k)\right\rangle$, negotiates a price for
one of them, and stores it as a new commitment.

Every scheduling step is a contract operation — projection, strengthening,
weakening, merging, separation, and a satisfiability query — so the feasible
starts are read out of the repaired contract rather than computed beside it.
A second signal, $plugged$, carries the vehicle's presence, which is what puts
the assumption slot to work: offers and stored agreements are conditional on
it.
Time and prices use exact rational arithmetic, and every contract is evaluated
at anchor $0$ over $[0,\infty)$.

$I$ may be a finite union of intervals and singletons: a singleton is a discrete
start, a continuous component denotes every start in it. Following
PROTOCOL.md's grouping convention, a disconnected region $S$ appears as
$\Diamond_S\varphi=\bigvee_\ell\Diamond_{S_\ell}\varphi$ and
$\Box_S\varphi=\bigwedge_\ell\Box_{S_\ell}\varphi$ over its connected
components. This extends PROTOCOL.md's integer start grid without new syntax.

Run the cells in order with the project environment, from the repository root
or `notebooks/`. All state is in memory; these helpers are a prototype, not a
package API.

In [ ]:
from dataclasses import dataclass, field
from fractions import Fraction as F
from itertools import combinations
from pathlib import Path
import sys

import portion as P
import interval_logic as il
from IPython.display import Markdown, display

notebook_dir = Path.cwd() if Path('steps.py').is_file() else Path.cwd() / 'notebooks'
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
from steps import reset_step, step, println

job = il.Signal('job')
plugged = il.Signal('plugged')
p = il.Proposition('p')
q = il.Proposition('q')


def fmt(value):
    if isinstance(value, P.Interval):
        return ' ∪ '.join(il.format_temporal_parameter(il.AbsoluteParameter(c))
                          for c in value) or '∅'
    if isinstance(value, tuple):
        return '(' + ', '.join(fmt(v) for v in value) + ')'
    return str(value)


def table(headers, rows):
    def line(values):
        return '| ' + ' | '.join(fmt(v).replace('|', r'\|') for v in values) + ' |'
    display(Markdown('\n'.join([line(headers), line(['---'] * len(headers)),
                                *(line(row) for row in rows)])))


def grouped(operator, body, region):
    '''Spell a region as one condition per connected component.'''
    connective = il.And if operator is il.Always else il.Or
    return connective(operator(body, component) for component in region).simplify(anchor=0)

## Contracts

A contract $\langle A\mid G\rangle$ is stored saturated, with $G$ replaced by
$G\lor\neg A$. Merging conjoins assumptions and guarantees; separation returns
the least contract that, merged with the other one, still refines this one.
`require_assumption` promotes an assumption to an explicit requirement, which is
how [PROTOCOL.md §3](../docs/PROTOCOL.md#3-repair-availability) forms
$\bar C^{\mathrm{post}}$.

There is one satisfiability procedure, `satisfiable` -- clause 1 of the
realization lemma, here `normalize` followed by `il.simplify(anchor=0)` failing
to reach $\bot$ -- and three contract-level entry points into it:

```python
consistent(c) = satisfiable(c.guarantee)
compatible(c) = satisfiable(c.assumption)
realizable(c) = satisfiable(c.assumption & c.guarantee)
```

`consistent` is the right question only while assumptions are trivial. A
saturated guarantee absorbs $\neg A$, so an *incompatible* contract is
consistent exactly because it is broken -- merging two commitments that cannot
both hold gives the guarantee $\top$. Steps 4 and 6 therefore ask `realizable`;
step 2 keeps `consistent`, since its candidate is unconditional. Note
`realizable(c) == consistent(c.require_assumption())`: promotion is the query
device. That is exact for the
reservation model — conflicting coverage of one signal is reduced to $\bot$ —
but in general an unreduced formula is evidence of nothing, so a richer model
needs a trajectory solver rather than this query.

`normalize` is the notebook's Boolean bookkeeping: conjunctive normal form
closed under resolution, with tautologies and subsumed clauses dropped, treating
temporal conditions as propositional leaves. It is exact but can blow up, and it
exists only to put the separation–merge result back into a readable shape. The
two temporal region rewrites this protocol needs — rewriting $\Box_I\Box_dp$ as
coverage of $I\oplus[0,d)$, and removing blocked times from a witness region —
now live in `il.simplify`; see
[CODE.md](../docs/CODE.md#parsing-formatting-rewriting-and-traversal).

In [ ]:
@dataclass(frozen=True)
class Contract:
    assumption: il.Formula
    guarantee: il.Formula

    def __post_init__(self):
        a = il.to_nnf(self.assumption)
        object.__setattr__(self, 'assumption', a)
        object.__setattr__(self, 'guarantee', il.to_nnf(il.as_formula(self.guarantee) | ~a))

    def __str__(self):
        return f'⟨ {self.assumption} ∣ {self.guarantee} ⟩'

    def merge(self, other):
        a = self.assumption & other.assumption
        return Contract(a, (self.guarantee & other.guarantee) | ~a)

    def separate(self, other):
        a, g, b, h = self.assumption, self.guarantee, other.assumption, other.guarantee
        return Contract((a & h) | ~(g & b), g & b)

    def require_assumption(self):
        return Contract(True, self.assumption & self.guarantee)

    def normalize(self):
        return Contract(normalize(self.assumption), normalize(self.guarantee))


def satisfiable(formula):
    '''Clause 1 of the realization lemma: does some trajectory satisfy it?'''
    return normalize(formula).simplify(anchor=0) != il.Boolean(False)


def consistent(contract):
    return satisfiable(contract.guarantee)


def compatible(contract):
    return satisfiable(contract.assumption)


def realizable(contract):
    '''Assumption and guarantee together -- the query conditional contracts need.

    `consistent` is not a weaker version of this. A saturated guarantee absorbs
    the negated assumption, so an *incompatible* contract is consistent exactly
    because it is broken: merging two commitments that cannot both hold gives
    the guarantee True. Steps 4 and 6 therefore ask `realizable`.
    '''
    return satisfiable(contract.assumption & contract.guarantee)


def _dual(literal):
    return il.to_nnf(~literal)


def _minimal_clauses(clauses):
    clauses = {c for c in clauses if not any(_dual(lit) in c for lit in c)}
    if frozenset() in clauses:
        return {frozenset()}
    return {c for c in clauses if not any(other < c for other in clauses)}


def _cnf(formula):
    # Temporal conditions are propositional leaves.
    if isinstance(formula, il.Boolean):
        return set() if formula.value else {frozenset()}
    if isinstance(formula, il.And):
        return _minimal_clauses(set().union(*(_cnf(c) for c in formula.operands)))
    if isinstance(formula, il.Or):
        result = {frozenset()}
        for child in formula.operands:
            result = _minimal_clauses({a | b for a in result for b in _cnf(child)})
        return result
    return {frozenset((formula,))}


def normalize(formula):
    '''Resolve and subsume to a fixed point, then simplify the regions.'''
    current = il.to_nnf(formula)
    while True:
        clauses = _cnf(current)
        while True:
            additions = set()
            for left, right in combinations(clauses, 2):
                for literal in left:
                    complement = _dual(literal)
                    if complement in right:
                        additions.add((left - {literal}) | (right - {complement}))
            reduced = _minimal_clauses(clauses | additions)
            if reduced == clauses:
                break
            clauses = reduced
        ordered = sorted(clauses, key=lambda c: tuple(sorted(map(str, c))))
        result = il.And(il.Or(sorted(c, key=str)) for c in ordered).simplify(anchor=0)
        if result == current:
            return result
        current = result

## Requests and commitments

A `Reservation` is a stored agreement: a positive job on one half-open
interval, held as the *conditional* contract that was accepted, with the
vehicle's presence as its assumption. Its `.allocation` is the unconditional
job-side abstraction that
[PROTOCOL.md §1](../docs/PROTOCOL.md#1-project-stored-reservations-to-the-requested-job)
projects to unavailability -- a reservation blocks a later request whether or
not its vehicle arrives, so steps 1-3 drop the assumption and steps 4 and 6
keep it. A `Request`
carries its job, duration, and start region. Each builds its contract once; the
protocol below then works on those contracts, and keeps the plain fields for the
concrete interval arithmetic of step 4.

In [ ]:
@dataclass(frozen=True)
class Reservation:
    job: int
    interval: P.Interval
    contract: Contract = field(init=False)

    def __post_init__(self):
        if type(self.job) is not int or self.job <= 0:
            raise ValueError('a reservation needs a positive job id')
        if not self.interval.atomic or self.interval.empty:
            raise ValueError('a reservation is one nonempty half-open interval')
        object.__setattr__(self, 'contract',
                           Contract(il.Always(plugged.equals(self.job), self.interval),
                                    il.Always(job.equals(self.job), self.interval)))

    @property
    def allocation(self):
        '''The unconditional job-side abstraction used for availability.

        A reservation blocks a later request whether or not its vehicle turns
        up, so steps 1-3 drop the presence assumption. Steps 4 and 6 use
        `.contract`, where it is retained.
        '''
        return Contract(True, il.Always(job.equals(self.job), self.interval))

    def agreement(self):
        '''The conditional service contract offered to the vehicle.

        The allocation in `.contract` is unconditional and is what blocks later
        requests; this is what the charger actually owes, and it is void where
        the vehicle is absent.
        '''
        return Contract(il.Always(plugged.equals(self.job), self.interval),
                        il.Always(job.equals(self.job), self.interval))

    @property
    def start(self):
        return self.interval.lower

    @property
    def duration(self):
        return self.interval.upper - self.interval.lower


@dataclass(frozen=True)
class Request:
    job: int
    duration: F
    starts: P.Interval
    contract: Contract = field(init=False)

    def __post_init__(self):
        if type(self.job) is not int or self.job <= 0:
            raise ValueError('a request needs a positive job id')
        if self.duration <= 0:
            raise ValueError('duration must be positive')
        if any(component.upper == P.inf for component in self.starts):
            raise ValueError('the start region must be bounded')
        object.__setattr__(self, 'contract', Contract(True, grouped(
            il.Eventually, il.Always(job.equals(self.job), self.duration), self.starts)))


def commitments(*reservations):
    '''The stored, conditional commitments -- what steps 4 and 6 check against.'''
    contract = Contract(True, True)
    for reservation in reservations:
        contract = contract.merge(reservation.contract)
    return contract


def allocations(*reservations):
    '''The unconditional availability abstraction -- what steps 1-3 consume.'''
    contract = Contract(True, True)
    for reservation in reservations:
        contract = contract.merge(reservation.allocation)
    return contract

## Steps 1–3: project, strengthen, repair

**[Project](../docs/PROTOCOL.md#1-project-stored-reservations-to-the-requested-job).**
Replace each $job=n$ atom by $p$ when $n$ is the requested job and $\neg p$
otherwise. Since the job is new, every commitment becomes $\Box_{R_\ell}\neg p$.
This forgets which other job holds the point, so the original commitments are
kept for step 4. Projection carries the two signals to two **independent**
propositions, `job` to $p$ and `plugged` to $q$. Collapsing both onto $p$ would
turn the vehicle's promise to be present into a claim about the charger's own
allocation, and the per-signal contradiction table would then report conflicts
that do not exist.

**[Strengthen](../docs/PROTOCOL.md#2-conjunctively-strengthen-the-request-and-test-contract-consistency).**
Turn each existential start alternative $\Diamond_{I_\ell}\Box_dp$ into the
universal requirement $\Box_{I_\ell}\Box_dp$, which `il.simplify` rewrites as
coverage of the envelope $I\oplus[0,d)$. Merged with the projected commitments,
an inconsistent candidate exposes a conflict anywhere in that envelope — it does
not yet mean the request is infeasible.

**[Repair](../docs/PROTOCOL.md#3-repair-availability).**
Weaken coverage to the existence of an available time, separate against the
projected commitments, merge back, and require the resulting assumption. The
repaired guarantee keeps the commitments alongside a witness region that
`il.simplify` has already stripped of the blocked times. `availability` reads
that region back out — it is the output of the repair, not a separate
calculation. When the candidate was consistent there is nothing to repair and
the weakened envelope is used directly.

`analyze` returns only the availability contract; everything else is logged.

In [ ]:
def project(contract, target_job):
    '''Carry `job` to p and `plugged` to q -- two independent propositions.

    Collapsing both onto p would be a category error, not an imprecision: it
    would turn the vehicle's promise to be present into a claim about the
    charger's own allocation, and the per-signal contradiction table would then
    report conflicts that do not exist.
    '''
    literal = il.NaturalLiteral(target_job)
    targets = ((job, p), (plugged, q))

    def visit(node):
        if isinstance(node, il.Equal):
            for signal, positive in targets:
                if signal in (node.left, node.right):
                    return positive if literal in (node.left, node.right) else ~positive
            return node
        children = tuple(visit(child) for child in node.children)
        return node._with_children(children) if children != node.children else node

    return Contract(visit(contract.assumption), visit(contract.guarantee))


def _terms(formula, node_type):
    return formula.operands if isinstance(formula, node_type) else (formula,)


def strengthen(request_contract):
    return Contract(True, il.And(
        il.Always(alternative.body, alternative.parameter)
        for alternative in _terms(request_contract.guarantee, il.Or)
    ))


def weaken(coverage_contract):
    return Contract(True, il.Or(
        il.Eventually(condition.body, condition.parameter)
        for condition in _terms(normalize(coverage_contract.guarantee), il.And)
    ))


def availability(contract):
    '''The witness region the contract still allows the requested job.'''
    region = P.empty()
    for term in _terms(normalize(contract.guarantee), il.And):
        for alternative in _terms(term, il.Or):
            if isinstance(alternative, il.Eventually) and alternative.body == p:
                region |= alternative.parameter.interval
    return region


def analyze(request, commitment):
    with step('Project the request and the commitments'):
        cmt = project(commitment, request.job)
        println(f'{cmt = !s}', '')
        req = project(request.contract, request.job)
        println(f'{req = !s}')

    with step('Strengthen and decide consistency'):
        strong = strengthen(req).normalize()
        println(f'{strong = !s}', '')
        candidate = cmt.merge(strong)
        println(f'{candidate = !s}', '')
        passes = consistent(candidate)
        println(f'{passes = !s}')

    with step('Derive the availability contract'):
        weak = weaken(strong)
        println(f'{weak = !s}', '')
        if passes:
            avail = weak
        else:
            with step('Separate, merge, and require the assumption'):
                separated = weak.separate(cmt)
                println(f'{separated = !s}', '')
                post = separated.merge(cmt).require_assumption().normalize()
                println(f'{post = !s}')
            avail = Contract(True, grouped(il.Eventually, p, availability(post)))
        println(f'{avail = !s}')
    return avail

## Step 4: concrete offers

[PROTOCOL.md §4](../docs/PROTOCOL.md#4-concretize-feasible-intervals) enumerates
the starts that keep the full duration inside the available region $J$:

$$
F=I\cap\{s:[s,s+d)\subseteq J\},
\qquad
\mathcal J=\{[s,s+d):s\in F\}.
$$

`erode` is that containment test. Its upper bound is closed at $b-d$, because a
window may end exactly where an excluded right endpoint begins, and a component
shorter than $d$ contributes no start at all.

$F$ is the whole feasible region, so representatives are chosen from it rather
than from a grid: every singleton survives, and a continuous component
contributes its left endpoint when included and otherwise its midpoint. No
nonempty component can therefore be missed. Each offer is checked against the request and against the original
commitments — the job vocabulary, not the projection — using `realizable`.
Consistency is not a weaker test at this site but the wrong one: an offer
overlapping another vehicle's booking makes the merged assumption $\bot$, hence
the saturated guarantee $\top$, so `consistent` would accept exactly the
windows it should reject.

In [ ]:
def erode(region, duration):
    '''Starts whose full half-open window fits inside the region.'''
    starts = P.empty()
    for component in region:
        if component.upper - duration >= component.lower:
            starts |= P.Interval.from_atomic(
                component.left, component.lower, component.upper - duration, P.CLOSED)
    return starts


def refines(offer, request):
    return (offer.job == request.job
            and offer.duration == request.duration
            and not (request.starts & erode(offer.interval, request.duration)).empty)


def synthesize(request, avail, commitment):
    with step('Compute the feasible starts'):
        feasible = request.starts & erode(availability(avail), request.duration)
        println(f'feasible = {fmt(feasible)}')

    with step('Generate the offer contracts'):
        offers = []
        for component in feasible:
            start = (component.lower if component.left == P.CLOSED
                     else (component.lower + component.upper) / 2)
            offer = Reservation(request.job, P.closedopen(start, start + request.duration))
            assert refines(offer, request)
            assert realizable(commitment.merge(offer.contract))
            println(f'{offer.contract!s}')
            offers.append(offer)
    return tuple(offers)

In [ ]:
reset_step()
stored = (Reservation(1, P.closedopen(F(2), F(3))),)
C_cmt = commitments(*stored)        # conditional: what steps 4 and 6 check against
C_alloc = allocations(*stored)      # unconditional: what steps 1-3 consume
request = Request(2, F(1), P.closedopen(F(1), F(5)))

with step('Intake: is the request compatible?'):
    println(f'compatible = {compatible(request.contract)}')
assert compatible(request.contract), 'a malformed request never reaches step 1'

C_avail = analyze(request, C_alloc)
offers = synthesize(request, C_avail, C_cmt)
table(['Offer agreement', 'Refines the request', 'Realizable with commitments'],
      [(offer.contract, refines(offer, request),
        realizable(C_cmt.merge(offer.contract))) for offer in offers])

⏺ Step 1: Project the request and the commitments
│ cmt = ⟨ ⊤ ∣ □_[2,3) ¬p ⟩
│                         
│ req = ⟨ ⊤ ∣ ◇_[1,5) □_1 p ⟩
╯
⏺ Step 2: Strengthen and decide consistency
│ strong = ⟨ ⊤ ∣ □_[1,6) p ⟩
│                           
│ candidate = ⟨ ⊤ ∣ □_[1,6) p ∧ □_[2,3) ¬p ⟩
│                                           
│ passes = False
╯
⏺ Step 3: Derive the availability contract
│ weak = ⟨ ⊤ ∣ ◇_[1,6) p ⟩
│                         
├ ⏺ Step 3.1: Separate, merge, and require the assumption
│ │ separated = ⟨ □_[1,6) ¬p ∨ □_[2,3) ¬p ∣ ◇_[1,6) p ⟩
│ │                                                    
│ │ post = ⟨ ⊤ ∣ (◇_[1,2) p ∨ ◇_[3,6) p) ∧ □_[2,3) ¬p ⟩
├─╯ 
│ avail = ⟨ ⊤ ∣ ◇_[1,2) p ∨ ◇_[3,6) p ⟩
╯
⏺ Step 4: Compute the feasible starts
│ feasible = [1,1] ∪ [3,5)
╯
⏺ Step 5: Generate the offer contracts
│ ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩
╯


| Offer contract | Refines the request | Consistent with commitments |
| --- | --- | --- |
| ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩ | True | True |
| ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩ | True | True |

## Presence: making the assumption slot load-bearing

Every contract above the offer layer is $\langle\top\mid G\rangle$, so the
assume-guarantee machinery carries no load. The charger's environment is the
vehicle, and the one thing the vehicle controls is whether it is connected:

$$
\textit{plugged}:X\to\mathbb N_0,
\qquad
\textit{charging}_k:=(\textit{job}=k)\land(\textit{plugged}=k).
$$

`plugged` is a *signal*, not a proposition, so it keeps the identity of the
connected vehicle and inherits the mutual exclusion the simplifier already
enforces on signal assignments. The station has **one connector**, which is
what makes that exclusion the right model: one vehicle at a time.

Commitments are therefore stored conditional, and $C^{\mathrm{cmt}}$'s
assumption is the conjunction of every stored presence window. Steps 1-3 use
the unconditional `.allocation` abstraction instead, so the availability
derivation is untouched.

Three consequences are checked below: projection must carry the two signals to
two propositions, `consistent` becomes the wrong query at the merge sites, and
an unplug event has a contractual reading rather than a procedural one.

In [ ]:
# 1. Projection carries two signals to two propositions; neither is consumed.
strength_2 = Contract(
    grouped(il.Eventually, il.Always(plugged.equals(2), F(1)), request.starts),
    grouped(il.Eventually, il.Always(job.equals(2), F(1)), request.starts))
with step('Project a contract that carries a presence assumption'):
    println(f'before: {strength_2!s}', '')
    println(f'after : {project(strength_2, 2)!s}')
projected = str(project(strength_2, 2))
assert ' q' in projected and ' p' in projected, 'job -> p and plugged -> q, independently'

In [ ]:
# 2. Consistency is not a weaker test here -- it is the wrong answer.
# [2,3) collides with job 1, so the two presence windows cannot both hold.
blocked = Reservation(2, P.closedopen(F(2), F(3)))
free    = Reservation(2, P.closedopen(F(3), F(4)))
table(['Window', 'compatible', 'consistent', 'realizable'],
      [(r.interval,
        compatible(C_cmt.merge(r.contract)),
        consistent(C_cmt.merge(r.contract)),
        realizable(C_cmt.merge(r.contract))) for r in (free, blocked)])

merged = C_cmt.merge(blocked.contract)
assert not compatible(merged)   # one connector cannot hold both vehicles
assert consistent(merged)       # ... so the saturated guarantee is True
assert not realizable(merged)   # the answer wanted: this would be a double booking

In [ ]:
# 3. An unplug at 3.5 violates the assumption; the guarantee is void from there.
accepted_agreement = free.contract
horizon = P.closedopen(F(0), F(10))
trace = il.Trace(
    universe=horizon,
    signals={'job':     il.NaturalSignal({2: P.closedopen(F(3), F(4))},    default=0),
             'plugged': il.NaturalSignal({2: P.closedopen(F(3), F(7, 2))}, default=0)})
holds = lambda f: il.satisfies(f, trace, F(0))
with step('Evaluate the accepted agreement on an unplug trace'):
    println(f'assumption  : {holds(accepted_agreement.assumption)}',
            f'saturated G : {holds(accepted_agreement.guarantee)}')
    println('reclaimable : [3.5,4)')
assert not holds(accepted_agreement.assumption) and holds(accepted_agreement.guarantee)

## Step 5: negotiate a price

[PROTOCOL.md §5](../docs/PROTOCOL.md#5-negotiate-interval-price-pairs) keeps the
offers fixed and bargains only over the total price $q$, with
$u^{\mathrm{EV}}=v(J)-q$ and $u^{\mathrm{CH}}=q-c(J)$. The charger proposes
first, each rejection switches roles, and targets concede by
$t_{r+1}=t_r(1-r/T)$ with $\lambda=1$, so round 0 leaves them unchanged.
Before the deadline the evaluator needs both its target and its floor; at the
deadline it needs only to exceed its floor.

The values below are PROTOCOL.md's illustration: costs $2$ and $3$ from tariffs
of $0.20$ and $0.30$ per kWh on $10$ kWh, EV values $8$ and $7$ from an
alternative costing $8$ with inconvenience $0$ and $1$, floors $1$, initial
targets $(6,4)$, and deadline $T=2$.

In [ ]:
def utility(party, offer, price, costs, values):
    return values[offer] - price if party == 'EV' else price - costs[offer]


def bargain(offers, costs, values, targets, floors, deadline=2):
    '''Alternating offers over fixed contracts, in exact arithmetic.'''
    offers = tuple(sorted(offers, key=lambda offer: offer.start))
    targets, history, proposer = dict(targets), [], 'CH'
    for r in range(deadline + 1):
        evaluator = 'EV' if proposer == 'CH' else 'CH'
        priced = [(offer, costs[offer] + targets[proposer] if proposer == 'CH'
                   else values[offer] - targets[proposer]) for offer in offers]
        priced = [(offer, price) for offer, price in priced
                  if utility(proposer, offer, price, costs, values) >= floors[proposer]]
        row = dict(round=r, targets=dict(targets), proposer=proposer, priced=priced)
        if not priced:
            history.append(row | dict(utilities=(), decision='No admissible offers'))
            return None, None, history, 'no_admissible_offers'
        utilities = tuple(utility(evaluator, offer, price, costs, values)
                          for offer, price in priced)
        best, price = priced[max(range(len(priced)), key=lambda i: utilities[i])]
        top = max(utilities)
        accepted = (top > floors[evaluator] if r == deadline
                    else top >= targets[evaluator] and top >= floors[evaluator])
        history.append(row | dict(utilities=utilities,
                                  decision='Accept' if accepted else 'Reject'))
        if accepted:
            return best, price, history, 'accepted'
        if r == deadline:
            return None, None, history, 'deadline_rejection'
        targets = {party: target * (1 - F(r, deadline)) for party, target in targets.items()}
        proposer = evaluator

In [18]:
cost = dict(zip(offers, (F(2), F(3)), strict=True))
value = dict(zip(offers, (F(8), F(7)), strict=True))
floors = {'EV': F(1), 'CH': F(1)}
accepted, agreed_price, rounds, status = bargain(
    offers, cost, value, {'EV': F(6), 'CH': F(4)}, floors)
with step('Negotiate over the fixed offers'):
    println(f'Result: {status}', f'Accepted agreement: {accepted.agreement()}')
    println(f'Total price: {agreed_price}')
table(['Round', 'Targets (EV, CH)', 'Proposer', 'Offers (interval, price)',
       'Evaluator utilities', 'Decision'],
      [(r['round'], (r['targets']['EV'], r['targets']['CH']), r['proposer'],
        '; '.join(f'({fmt(offer.interval)}, {price})' for offer, price in r['priced']),
        r['utilities'], r['decision']) for r in rounds])
assert (accepted, agreed_price) == (offers[0], F(4))

⏺ Step 6: Negotiate over the fixed offers
│ Result: accepted                          
│ Accepted contract: ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ Total price: 4
╯


| Round | Targets (EV, CH) | Proposer | Offers (interval, price) | Evaluator utilities | Decision |
| --- | --- | --- | --- | --- | --- |
| 0 | (6, 4) | CH | ([1,2), 6); ([3,4), 7) | (2, 0) | Reject |
| 1 | (6, 4) | EV | ([1,2), 2); ([3,4), 1) | (0, -2) | Reject |
| 2 | (3, 2) | CH | ([1,2), 4); ([3,4), 5) | (4, 2) | Accept |

## Step 6: validate and store

[PROTOCOL.md §6](../docs/PROTOCOL.md#6-validate-and-store-the-accepted-agreement)
revalidates the accepted contract against the current request and commitments
before inserting it, and stores the price as metadata beside the reservation.

This helper is sequential and in memory. A real service must validate and insert
atomically against one snapshot of the request, commitments, and model inputs,
restarting the scheduling when they have moved:

```text
snapshot := current request contract, commitment contract, and model inputs
offers := synthesize(analyze(snapshot))
if offers empty: end without agreement
agreement := bargain(offers)
if no agreement: end without agreement
atomically:
    if current inputs differ from snapshot: restart scheduling
    if the accepted contract fails refinement or consistency: restart scheduling
    merge the accepted contract into the commitments and store the price
```

A price rejection ends the run without agreement; only changed inputs force a
rescheduling. A vehicle that never arrives is a third case, and a different one:
the assumption of the stored agreement is violated, so the guarantee is void
from that instant and the charger may truncate the allocation and return the
tail to availability. The strengthened coverage, the repaired availability, and the
unaccepted offers are all temporary.

In [19]:
def commit(commitment, request, offer, price):
    if offer is None:
        return 'no_agreement', commitment, None
    if not refines(offer, request) or not realizable(commitment.merge(offer.contract)):
        return 'restart', commitment, None
    return 'committed', commitment.merge(offer.contract).normalize(), (offer, price)

In [20]:
status, C_committed, record = commit(C_cmt, request, accepted, agreed_price)
with step('Revalidate and store the accepted contract'):
    println(f'Status: {status}')
    println(f"EV utility: {utility('EV', accepted, agreed_price, cost, value)}")
    println(f'Charger utility: {utility("CH", accepted, agreed_price, cost, value)}')
    later = allocations(*stored, accepted)
    println(f'Availability seen by a later job 3: {project(later, 3).normalize()!s}')
stored_offer, stored_price = record
table(['Stored job', 'Interval', 'Price'],
      [(stored_offer.job, stored_offer.interval, stored_price)])
assert status == 'committed'
assert (stored_offer.job, stored_offer.interval, stored_price) == (2, P.closedopen(1, 2), F(4))
assert utility('EV', accepted, agreed_price, cost, value) == 4
assert utility('CH', accepted, agreed_price, cost, value) == 2

⏺ Step 7: Revalidate and store the accepted contract
│ Status: committed                                            
│ Updated commitment: ⟨ ⊤ ∣ □_[1,2) job = 2 ∧ □_[2,3) job = 1 ⟩
│ EV utility: 4
│ Charger utility: 2
│ Projection for a later job 3: ⟨ ⊤ ∣ □_[1,3) ¬p ⟩
╯


| Stored job | Interval | Price |
| --- | --- | --- |
| 2 | [1,2) | 4 |

## Validation — independent interval oracle

The interval calculations in this section are **test expectations only**. They
are never called by `analyze`, `consistent`, `synthesize`, bargaining, or commit.
The oracle computes forbidden start regions directly from independent raw test
fixtures: overlap with a reservation $[a,b)$ forbids starts in $(a-d,b)$.
This independently checks both boundaries without using the production erosion
or envelope helpers. Every fixture is converted to contracts before scheduling.

In [ ]:
# Scheduling fixtures and independent oracle
def oracle_starts(starts, duration, raw_reservations):
    result = starts
    for _, interval in raw_reservations:
        result = result - P.open(interval.lower - duration, interval.upper)
    return result


def contracts_from_fixture(raw_reservations):
    contract = Contract(True, True)
    for identifier, interval in raw_reservations:
        contract = contract.merge(reservation(identifier, interval))
    return contract


fixtures = [
    ('Canonical discrete repair', start_grid(4), F(1), ((1, P.closedopen(F(3, 2), 3)),)),
    ('No conflict', start_grid(4), F(1), ((1, P.closedopen(6, 8)),)),
    ('No commitments', start_grid(4), F(1), ()),
    ('Empty request', P.empty(), F(1), ()),
    ('Fully blocked', start_grid(4), F(1), ((1, P.closedopen(1, 5)),)),
    ('Short fragment', P.singleton(1), F(1), ((1, P.closedopen(F(3, 2), 2)),)),
    ('Disconnected envelope', start_grid(3), F(1, 2), ()),
    ('Endpoint adjacency', start_grid(3), F(1), ((1, P.closedopen(1, 3)),)),
    ('Continuous repair', P.closed(1, 4), F(1), ((1, P.closedopen(F(3, 2), 3)),)),
    ('Open left representative', P.openclosed(3, 4), F(1), ((1, P.closedopen(1, 3)),)),
    ('Open right start boundary', P.closedopen(3, 4), F(1), ()),
    ('Mixed starts', P.singleton(1) | P.open(3, 4) | P.singleton(5), F(1), ()),
    ('Multiple reservations', P.closed(0, 6), F(1),
     ((1, P.closedopen(1, 2)), (3, P.closedopen(4, 5)))),
    ('Only one exact-fit window', P.closed(0, 3), F(1),
     ((1, P.closedopen(0, 1)), (3, P.closedopen(2, 4)))),
    ('Open boundary excludes exact fit', P.openclosed(1, 3), F(1),
     ((1, P.closedopen(0, 1)), (3, P.closedopen(2, 4)))),
]
fixture_results, fixture_rows = {}, []
for label, starts, duration, raw in fixtures:
    req = Request(2, duration, starts).contract
    cmt = contracts_from_fixture(raw)
    result = analyze(req, cmt)
    expected = oracle_starts(starts, duration, raw)
    assert result.synthesis.feasible_starts == expected, label
    assert bool(result.synthesis.offers) == (not expected.empty), label
    # One output per component, preserving every singleton component.
    assert len(result.synthesis.offers) == len(tuple(expected)), label
    for component, offer in zip(expected, result.synthesis.offers, strict=True):
        view = offer_view(offer)
        assert view.start in component and view.duration == duration
        assert refines_request(offer, req) and consistent(cmt.merge(offer))
    assert consistent(cmt.merge(req)) == (not expected.empty), label
    fixture_results[label] = result
    fixture_rows.append((label, starts, expected,
                         tuple(offer_view(c).interval for c in result.synthesis.offers)))
table(['Case', 'Input starts', 'Contract-derived feasible starts', 'Selected offers'], fixture_rows)
assert fixture_results['No conflict'].separated is None
assert fixture_results['Canonical discrete repair'].separated is not None
assert fixture_results['Empty request'].candidate is None
assert tuple(offer_view(c).start for c in fixture_results['Open left representative'].synthesis.offers) == (F(7, 2),)
assert fixture_results['Only one exact-fit window'].synthesis.feasible_starts == P.singleton(1)
assert fixture_results['Open boundary excludes exact fit'].synthesis.feasible_starts.empty
short = fixture_results['Short fragment']
assert not availability_view(short.availability)[1].empty and not short.synthesis.offers

⏺ Step 7: Project Request and Commitements
│ cmt_p = Contract(assumption=Boolean(value=True), guarantee=Always(body=Not(operand=Proposition(name='p')), parameter=AbsoluteParameter(interval=[Fraction(3, 2),Fraction(3, 1)))))
│                                                                                                                                                                                   
│ req_p = Contract(assumption=Boolean(value=True), guarantee=Or(operands=(Eventually(body=Always(body=Proposition(name='p'), parameter=RelativeParameter(duration=Fraction(1, 1), include_endpoint=False)), parameter=AbsoluteParameter(interval=[Fraction(1, 1)])), Eventually(body=Always(body=Proposition(name='p'), parameter=RelativeParameter(duration=Fraction(1, 1), include_endpoint=False)), parameter=AbsoluteParameter(interval=[Fraction(2, 1)])), Eventually(body=Always(body=Proposition(name='p'), parameter=RelativeParameter(duration=Fraction(1, 1), include_endpoint=False)), parameter=Absolut

| Case | Input starts | Contract-derived feasible starts | Selected offers |
| --- | --- | --- | --- |
| Canonical discrete repair | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | [3,3] ∪ [4,4] | ([3,4), [4,5)) |
| No conflict | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | ([1,2), [2,3), [3,4), [4,5)) |
| No commitments | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | ([1,2), [2,3), [3,4), [4,5)) |
| Empty request | ∅ | ∅ | () |
| Fully blocked | [1,1] ∪ [2,2] ∪ [3,3] ∪ [4,4] | ∅ | () |
| Short fragment | [1,1] | ∅ | () |
| Disconnected envelope | [1,1] ∪ [2,2] ∪ [3,3] | [1,1] ∪ [2,2] ∪ [3,3] | ([1,1.5), [2,2.5), [3,3.5)) |
| Endpoint adjacency | [1,1] ∪ [2,2] ∪ [3,3] | [3,3] | ([3,4)) |
| Continuous repair | [1,4] | [3,4] | ([3,4)) |
| Open left representative | (3,4] | (3,4] | ([3.5,4.5)) |
| Open right start boundary | [3,4) | [3,4) | ([3,4)) |
| Mixed starts | [1,1] ∪ (3,4) ∪ [5,5] | [1,1] ∪ (3,4) ∪ [5,5] | ([1,2), [3.5,4.5), [5,6)) |
| Multiple reservations | [0,6] | [0,0] ∪ [2,3] ∪ [5,6] | ([0,1), [2,3), [5,6)) |
| Only one exact-fit window | [0,3] | [1,1] | ([1,2)) |
| Open boundary excludes exact fit | (1,3] | ∅ | () |

### Boolean identities, normalization, and refinement

Check the saturated separation–merge specialization on all 16 independent
Boolean valuations, then compare the raw and normalized temporal expressions
using both `satisfies` and `denote`. The finite test universe contains every
relevant window; these are independent checks of the fixed-context rules.

For a concrete offer, refinement holds exactly when it guarantees the requested
job throughout some allowed full-duration window. Negative cases below test
wrong jobs, disallowed starts, and insufficient coverage. Constructed traces
illustrate this supported implication without claiming arbitrary refinement.

In [ ]:
# Boolean algebra and trajectory checks
w0, w1, k0, k1 = (il.Proposition(name) for name in ('w0', 'w1', 'k0', 'k1'))
W, K = w0 | w1, k0 & k1
symbolic_raw = require_assumption(Contract(True, W).separate(Contract(True, K)).merge(Contract(True, K)))
boolean_result = _boolean_normalize(symbolic_raw.guarantee)
assert boolean_result == (K & W).simplify(anchor=0)
for values in product((False, True), repeat=4):
    valuation = {name: P.singleton(0) if value else P.empty()
                 for name, value in zip(('w0', 'w1', 'k0', 'k1'), values)}
    trace = il.Trace(P.singleton(0), propositions=valuation)
    assert il.satisfies(symbolic_raw.guarantee, trace, 0) == il.satisfies(boolean_result, trace, 0)
print('Separation–merge Boolean identity checked on all 16 independent valuations.')

trace_universe = P.closed(0, 10)
normalization_pairs = [(analysis.post_raw.guarantee, analysis.post.guarantee)]
for starts in (P.closed(1, 3), P.open(1, 3), P.singleton(1) | P.openclosed(3, 4)):
    projected = project(Request(2, F(1), starts).contract, 2)
    strengthened = strengthen_request(projected)
    normalization_pairs.append((strengthened.guarantee, normalize(strengthened).guarantee))
# A disconnected repaired envelope with several commitments exercises CNF and
# component pruning together, rather than only the single-reservation identity.
multi_req = Request(2, F(1, 2), P.singleton(1) | P.closed(3, 4)).contract
multi_cmt = reservation(1, P.closedopen(F(5, 4), F(3, 2))).merge(
    reservation(3, P.closedopen(F(7, 2), F(15, 4))))
multi = analyze(multi_req, multi_cmt)
normalization_pairs.append((multi.post_raw.guarantee, multi.post.guarantee))
for region in (P.empty(), trace_universe, P.closedopen(3, 5), P.singleton(3),
               P.closedopen(1, 4), P.open(1, 4), P.closed(1, 4),
               P.closedopen(1, 2) | P.open(3, 5),
               P.closedopen(1, F(5, 4)) | P.closedopen(F(15, 4), F(9, 2))):
    trace = il.Trace(trace_universe, propositions={'p': region})
    for before, after in normalization_pairs:
        assert il.satisfies(before, trace, 0) == il.satisfies(after, trace, 0)
        assert il.denote(before, trace, 0) == il.denote(after, trace, 0)
        assert normalize(Contract(True, after)).guarantee == after

for offered, requested, expected in (
    (reservation(2, P.closedopen(3, 4)), C_req, True),
    (reservation(3, P.closedopen(3, 4)), C_req, False),
    (reservation(2, P.closedopen(F(7, 2), F(9, 2))), C_req, False),
    (reservation(2, P.closedopen(3, F(7, 2))), C_req, False),
    (reservation(2, P.closedopen(0, 5)), C_req, True),
    (reservation(2, P.closedopen(3, 4)), Request(2, F(1), P.open(3, 4)).contract, False),
    (reservation(2, P.closedopen(F(7, 2), F(9, 2))),
     Request(2, F(1), P.open(3, 4)).contract, True),
    (reservation(2, P.closedopen(3, 4)), Request(2, F(1), P.empty()).contract, False),
):
    assert refines_request(offered, requested) == expected
    # A trace true for exactly the offered allocation independently witnesses
    # request satisfaction (or a counterexample to the claimed refinement).
    view = offer_view(offered)
    trace = il.Trace(trace_universe, signals={
        'job': il.NaturalSignal({view.job: view.interval}, default=0)
    })
    assert il.satisfies(offered.guarantee, trace, 0)
    assert il.satisfies(requested.guarantee, trace, 0) == expected

for offer in alternatives:
    decoded = offer_view(offer)
    trace = il.Trace(trace_universe, propositions={'p': decoded.interval}, signals={
        'job': il.NaturalSignal({1: offer_view(C_cmt).interval, 2: decoded.interval}, default=0)
    })
    assert il.satisfies(C_req.guarantee, trace, 0)
    assert il.satisfies(C_cmt.merge(offer).guarantee, trace, 0)
    assert il.satisfies(analysis.post.guarantee, trace, 0)
    assert not il.satisfies(analysis.strengthened.guarantee, trace, 0)
availability_only = il.Trace(trace_universe, propositions={'p': trace_universe})
assert il.satisfies(analysis.availability.guarantee, availability_only, 0)
assert not il.satisfies(analysis.post.guarantee, availability_only, 0)
print('Normalization truth/masks and offer witness traces checked; availability is strictly weaker.')

Separation–merge Boolean identity checked on all 16 independent valuations.
⏺ Step 49: Project Request and Commitements
│ cmt_p = Contract(assumption=Boolean(value=True), guarantee=And(operands=(Always(body=Not(operand=Proposition(name='p')), parameter=AbsoluteParameter(interval=[Fraction(5, 4),Fraction(3, 2)))), Always(body=Not(operand=Proposition(name='p')), parameter=AbsoluteParameter(interval=[Fraction(7, 2),Fraction(15, 4)))))))
│                                                                                                                                                                                                                                                                                                                            
│ req_p = Contract(assumption=Boolean(value=True), guarantee=Or(operands=(Eventually(body=Always(body=Proposition(name='p'), parameter=RelativeParameter(duration=Fraction(1, 2), include_endpoint=False)), parameter=AbsoluteParameter(interval=[Fr

### Verify contract dependencies and explicit failures

Change the actual postcondition, request, or commitment contract and synthesize
again. No reservation records or cached envelope accompany these calls.
Unsupported syntax raises `UnsupportedContractError`; malformed inputs raise
`ValueError`. Neither becomes assumed success.

In [ ]:
# Contract dependency and unsupported-input checks
tighter_post = normalize(Contract(True, analysis.post.guarantee & il.Always(~p, P.closedopen(4, 5))))
tighter = synthesize(C_req, extract_availability(tighter_post), C_cmt)
assert tuple(offer_view(c).start for c in tighter.offers) == (F(3),)
blocked_post = normalize(Contract(True, analysis.post.guarantee & il.Always(~p, P.closedopen(3, 5))))
assert not synthesize(C_req, extract_availability(blocked_post), C_cmt).offers
changed_req_contract = Request(2, F(1), P.singleton(4)).contract
assert tuple(offer_view(c).start for c in synthesize(
    changed_req_contract, analysis.availability, C_cmt).offers) == (F(4),)
changed_cmt_contract = reservation(1, P.closedopen(F(3, 2), 4))
assert tuple(offer_view(c).start for c in synthesize(
    C_req, analysis.availability, changed_cmt_contract).offers) == (F(4),)
# Synthesis receives no raw reservations, horizon, envelope, or blocked set.
table(['Changed contract input', 'Result'], [
    ('Postcondition forbids [4,5)', 'Only start 3'),
    ('Postcondition forbids [3,5)', 'No offers'),
    ('Request allows only start 4', 'Only start 4'),
    ('Commitment extends to 4', 'Only start 4'),
])

unsupported_calls = [
    lambda: consistent(Contract(True, il.Until(p, p, 1))),
    lambda: consistent(Contract(p, p)),
    lambda: consistent(Contract(True, il.Always(p, 1))),
    lambda: consistent(Contract(True, il.Eventually(p, P.closed(1, 2)) & il.Eventually(p, P.closed(3, 4)))),
    lambda: consistent(Contract(True, il.Always(p, P.closed(1, 2)) & il.Always(job.equals(1), P.closed(3, 4)))),
    lambda: project(Contract(True, il.Always(job.equals(il.Signal('other')), P.closed(1, 2))), 2),
    lambda: project(Contract(True, il.Always(~job.equals(1), P.closed(1, 2))), 2),
    lambda: extract_availability(Contract(True, il.Always(~p, P.closed(1, 2)))),
    lambda: extract_availability(Contract(True, il.Eventually(~p, P.closed(1, 2)))),
    lambda: Request(2, F(1), P.closedopen(0, P.inf)),
]
for call in unsupported_calls:
    try:
        call()
    except UnsupportedContractError:
        pass
    else:
        raise AssertionError('unsupported input was silently accepted')
for bad_request in (lambda: Request(0, F(1), P.singleton(1)),
                    lambda: Request(2, F(0), P.singleton(1)),
                    lambda: Request(2, F(1), P.closed(-1, 2)),
                    lambda: analyze(Request(1, F(1), P.singleton(4)).contract, C_cmt),
                    lambda: analyze(C_req, C_cmt.merge(reservation(3, P.closedopen(2, 4))))):
    try:
        bad_request()
    except ValueError:
        pass
    else:
        raise AssertionError('invalid reservation-model input was accepted')
print('Unsupported syntax and invalid inputs fail explicitly.')

| Changed contract input | Result |
| --- | --- |
| Postcondition forbids [4,5) | Only start 3 |
| Postcondition forbids [3,5) | No offers |
| Request allows only start 4 | Only start 4 |
| Commitment extends to 4 | Only start 4 |

Unsupported syntax and invalid inputs fail explicitly.


### Bargaining and unchanged-state outcomes

Check all three rounds, unchanged round-0 targets, alternating roles, proposer
floors, strict deadline acceptance, and earliest-start ties. Failed negotiation,
changed snapshots, and invalid accepted allocations leave commitments unchanged.
These sequential checks establish no concurrency guarantee.

In [ ]:
# Negotiation and state-update checks
assert [row['proposer'] for row in rounds] == ['CH', 'EV', 'CH']
assert [row['targets'] for row in rounds] == [initial_targets, initial_targets, {'EV': F(3), 'CH': F(2)}]
assert [tuple(o.price for o in row['offers']) for row in rounds] == [(F(6), F(7)), (F(2), F(1)), (F(4), F(5))]
assert [row['utilities'] for row in rounds] == [(F(2), F(0)), (F(0), F(-2)), (F(4), F(2))]
for row in rounds:
    assert tuple(o.contract for o in row['offers']) == alternatives
    assert all(utility(row['proposer'], o, cost, value) >= floors[row['proposer']] for o in row['offers'])
assert acceptable(F(1), F(1), F(1), 0, 2)
assert not acceptable(F(1), F(0), F(1), 2, 2)
no_offer, _, no_offer_status = bargain_illustration(
    alternatives, cost, value, {'EV': F(6), 'CH': F(0)}, floors)
rejected, _, rejected_status = bargain_illustration(
    alternatives[:1], {alternatives[0]: F(2)}, {alternatives[0]: F(4)}, initial_targets, floors)
assert no_offer is None and no_offer_status == 'no_admissible_offers'
assert rejected is None and rejected_status == 'deadline_rejection'
tie, _, _ = bargain_illustration(tuple(reversed(alternatives)),
    {c: F(2) for c in alternatives}, {c: F(8) for c in alternatives}, {'EV': F(1), 'CH': F(1)}, floors)
assert tie.contract == alternatives[0]

state_cases = [
    ('No admissible offers', C_cmt, C_req, no_offer, 'no_agreement'),
    ('Deadline rejection', C_cmt, C_req, rejected, 'no_agreement'),
    ('Commitments changed', C_cmt.merge(reservation(3, P.closedopen(3, 4))), C_req, accepted, 'restart'),
    ('Request changed', C_cmt, changed_req_contract, accepted, 'restart'),
    ('Infeasible selected contract', C_cmt, C_req, PricedOffer(reservation(2, P.closedopen(1, 2)), F(4)), 'restart'),
    ('Wrong purchased duration', C_cmt, C_req, PricedOffer(reservation(2, P.closedopen(3, 5)), F(4)), 'restart'),
]
state_rows = []
for label, current_cmt, current_req, agreement, expected in state_cases:
    status, updated, record = validate_commit_demo(current_cmt, current_req, C_cmt, C_req, agreement)
    assert status == expected and updated == current_cmt and record is None
    state_rows.append((label, status, 'Unchanged'))
assert normalize(project(C_committed, 3)) == Contract(True, il.Always(~p, P.closedopen(F(3, 2), 4)))
table(['Case', 'Status', 'Commitments'], state_rows)
print('All contract-driven protocol checks passed. General dynamics and atomic storage remain unsupported.')

| Case | Status | Commitments |
| --- | --- | --- |
| No admissible offers | no_agreement | Unchanged |
| Deadline rejection | no_agreement | Unchanged |
| Commitments changed | restart | Unchanged |
| Request changed | restart | Unchanged |
| Infeasible selected contract | restart | Unchanged |
| Wrong purchased duration | restart | Unchanged |

All contract-driven protocol checks passed. General dynamics and atomic storage remain unsupported.
